# HotpotQA Small Reproduction

This notebook runs the HotpotQA ReAct setting on 10 dev examples to save API tokens.

## 1. API and Model

Before running this notebook, set your API key in the terminal that launches Jupyter:

PowerShell:
```powershell
$env:OPENAI_API_KEY="your_api_key"
```

Anaconda Prompt / CMD:
```bat
set OPENAI_API_KEY=your_api_key
```

In [1]:
import os
from openai import OpenAI

MODEL = "gpt-4.1-mini"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def llm(prompt, stop=["\n"], temperature=0, max_tokens=100):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
        stop=stop,
    )
    return response.choices[0].message.content or ""

## 2.1 ReAct
### 2.1.1 Environment

`WikiEnv` executes `search[]`, `lookup[]`, and `finish[]`. `HotPotQAWrapper` provides questions and scoring.

In [2]:
import requests # 防止遇到 timeout 时报错
import wikienv, wrappers

env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env, folder="trajs", file_id="hotpotqa_react_10")

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1
    raise RuntimeError(f"Wikipedia request timed out after {attempts} attempts")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### 2.1.2 ReAct Prompt

The paper uses 6 manually written HotpotQA ReAct trajectories as few-shot exemplars.

In [3]:
import json

with open("./prompts/prompts_naive.json", "r", encoding="utf-8") as f:
    prompt_dict = json.load(f)

webthink_examples = prompt_dict["webthink_simple6"]
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webthink_prompt = instruction + webthink_examples

print(webthink_prompt[:1000])

Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.

Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought 2: It does not mention the eastern sec

### 2.1.3 ReAct Inference
#### 源代码已经不适用于新模型 gpt-4.1-mini

假设程序现在已经跑到第 3 轮了，当前 prompt 末尾长这样：
```text
Question: Who is older Danny Green or James Worthy?

Thought 1: I need to find Danny Green's birth date.
Action 1: Search[Danny Green]
Observation 1: Danny Green is an American basketball player born June 22, 1987.

Thought 2: I need to find James Worthy's birth date.
Action 2: Search[James Worthy]
Observation 2: James Worthy is an American basketball player born February 27, 1961.

Thought 3:
```
然后把这个 prompt 发给模型

#### 老 completion 模型的典型行为
老模型 `text-davinci-002` 更像接着写文本，它通常只补后面的内容：
```text
 James Worthy was born in 1961 and Danny Green was born in 1987, so James Worthy is older.
Action 3: Finish[James Worthy]
```
拼回去就是：
```text
Thought 3: James Worthy was born in 1961 and Danny Green was born in 1987, so James Worthy is older.
Action 3: Finish[James Worthy]
```
这正好符合原代码预期

#### 新 chat 模型的典型行为

新模型更像在回答用户请求，不是纯粹续写

它看到 prompt 里一堆示例都是：
```text
Question: ...
Thought 1: ...
Action 1: ...
Observation 1: ...
```
于是它可能认为我应该输出一个完整格式的下一步，所以它输出：
```text
Thought 3: James Worthy was born in 1961 and Danny Green was born in 1987, so James Worthy is older.
Action 3: Finish[James Worthy]
```
但程序前面已经给了它 `Thought 3:`，于是拼起来就变成：
```text
Thought 3: Thought 3: James Worthy was born...
Action 3: Finish[James Worthy]
```

更糟的是，有时候新模型会像重新开始答一道题一样输出：
```text
Thought 1: I need to compare their birth dates.
Action 1: Finish[James Worthy]
```
于是拼起来变成：
```text
Thought 3: Thought 1: I need to compare their birth dates.
Action 1: Finish[James Worthy]
```
程序现在是第 3 轮，原代码只找 `Action 3:`

但模型给的是 `Action 1:`，于是原代码找不到 `Action 3:`，就解析失败

#### 对代码做如下改动
- 加了 import re
- 把原来严格依赖 Action {i}: 的解析方式换成正则
- 现在即使模型输出 Action 1: Search[...] 出现在第 3 轮，也会提取真正的 Search[...]
- fallback 里也会去掉多余的 Action 1: 前缀

Each question can use at most 7 ReAct steps, matching the HotpotQA setting described in the paper.

In [4]:
import re

def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt = prompt + question + "\n"
    n_calls, n_badcalls = 0, 0
    done = False
    info = {}
    r = 0

    for i in range(1, 8):
        n_calls += 1
        thought_action = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"])
        thought_action = thought_action.strip()

        prefix = f"Thought {i}:"
        if thought_action.startswith(prefix):
            thought_action = thought_action[len(prefix):].strip()

        action_match = re.search(
            r"Action\s*\d+\s*:\s*(Search\[.*?\]|Lookup\[.*?\]|Finish\[.*?\])",
            thought_action,
            re.I | re.S,
        )
        if action_match:
            thought = thought_action[:action_match.start()].strip()
            action = action_match.group(1).strip()
        else:
            print("Could not parse thought/action:", thought_action)
            n_badcalls += 1
            n_calls += 1
            thought = thought_action.strip().split("\n")[0]
            action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=["\n"]).strip()
            action = re.sub(r"^Action\s*\d+\s*:\s*", "", action).strip()

        if not action:
            action = "finish[]"

        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace("\n", "")
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str

        if to_print:
            print(step_str)
        if done:
            break

    if not done:
        obs, r, done, info = step(env, "finish[]")

    if to_print:
        print(info, "\n")
    info.update({"n_calls": n_calls, "n_badcalls": n_badcalls, "traj": prompt})
    return r, info

### 2.1.4 Run 10 Examples

The paper runs 500 randomly selected dev examples. This notebook runs only 10 examples with the same random seed.

- 把每次 webthink() 的完整轨迹 info["traj"] 写到 `hotpotqa_react_raw_10.txt`
- 跑完后把 infos 写到 `hotpotqa_react_10.json`

In [5]:
import random
import time
import os
import json

N_EXAMPLES = 10
idxs = list(range(len(env)))
random.Random(233).shuffle(idxs)

os.makedirs("trajs", exist_ok=True)

rs = []
infos = []
old_time = time.time()

with open("trajs/hotpotqa_react_raw_10.txt", "w", encoding="utf-8") as raw_f:
    for i in idxs[:N_EXAMPLES]:
        r, info = webthink(i, to_print=True)
        rs.append(info["em"])
        infos.append(info)

        print("correct:", sum(rs), "seen:", len(rs), "EM:", sum(rs) / len(rs), "avg_time:", (time.time() - old_time) / len(rs))
        print("-----------\n")

        raw_f.write(info["traj"])
        raw_f.write("\n")
        raw_f.write(str(info))
        raw_f.write("\n-----------\n\n")

    print("Final EM:", sum(rs) / len(rs))
    raw_f.write("Final EM: " + str(sum(rs) / len(rs)) + "\n")

with open("trajs/hotpotqa_react_10.json", "w", encoding="utf-8") as f:
    json.dump(infos, f, indent=2, ensure_ascii=False)

len(infos)

3687 Question: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
Thought 1: The question asks about a movie that actress Irene Jacob completed before an American action crime thriller film directed by Stuart Bird. I need to find the name of the American action crime thriller film directed by Stuart Bird first, then find what movie Irene Jacob completed before that film.
Action 1: Search[Stuart Bird]
Observation 1: 

Thought 2: The search for Stuart Bird returned no information. I need to try searching for the American action crime thriller film directed by Stuart Bird instead, to find its title.
Action 2: Search[American action crime thriller film directed by Stuart Bird]
Observation 2: 

Thought 3: The search for Stuart Bird and the description of the film directed by him returned no information. I need to try searching for Irene Jacob to see if there is any mention of a film related to Stuart Bird or an American action

10

In [6]:
total_em = sum(info["em"] for info in infos) / len(infos)
total_f1 = sum(info["f1"] for info in infos) / len(infos)

print("Total EM:", total_em)
print("Total F1:", total_f1)

Total EM: 0.1
Total F1: 0.1


## 2.2 CoT baseline

Same 10 HotpotQA examples, no Wikipedia actions.

In [ ]:
import os
import json
import random
import time
import re

N_EXAMPLES = 10
idxs = list(range(len(env)))
random.Random(233).shuffle(idxs)

os.makedirs("trajs", exist_ok=True)

cot_prompt = prompt_dict["cotqa_simple6"]

def cot_answer(idx, to_print=True):
    question = env.reset(idx=idx)
    gold = env.env.data[env.env.data_idx][1]
    prompt = cot_prompt + question + "\nThought:"
    completion = llm(prompt, stop=["\nQuestion:"], temperature=0, max_tokens=150).strip()
    traj = prompt + completion

    answer_match = re.search(r"Answer\s*:\s*(.*)", completion, re.I | re.S)
    if answer_match:
        pred = answer_match.group(1).strip().split("\n")[0].strip()
    else:
        pred = completion.strip().split("\n")[-1].strip()

    em = wrappers.normalize_answer(pred) == wrappers.normalize_answer(gold)
    f1 = wrappers.f1_score(pred, gold)[0]

    info = {
        "question_idx": idx,
        "question": question.replace("Question: ", "", 1),
        "answer": pred,
        "gt_answer": gold,
        "em": em,
        "f1": f1,
        "traj": traj,
    }

    if to_print:
        print(idx, question)
        print(completion)
        print(info, "\n")

    return int(em), info

cot_rs = []
cot_infos = []
old_time = time.time()

with open("trajs/hotpotqa_cot_raw_10.txt", "w", encoding="utf-8") as raw_f:
    for i in idxs[:N_EXAMPLES]:
        r, info = cot_answer(i, to_print=True)
        cot_rs.append(info["em"])
        cot_infos.append(info)

        print("correct:", sum(cot_rs), "seen:", len(cot_rs), "EM:", sum(cot_rs) / len(cot_rs), "avg_time:", (time.time() - old_time) / len(cot_rs))
        print("-----------\n")

        raw_f.write(info["traj"])
        raw_f.write("\n")
        raw_f.write(str(info))
        raw_f.write("\n-----------\n\n")

    print("CoT Final EM:", sum(cot_rs) / len(cot_rs))
    print("CoT Final F1:", sum(info["f1"] for info in cot_infos) / len(cot_infos))
    raw_f.write("Final EM: " + str(sum(cot_rs) / len(cot_rs)) + "\n")
    raw_f.write("Final F1: " + str(sum(info["f1"] for info in cot_infos) / len(cot_infos)) + "\n")

with open("trajs/hotpotqa_cot_10.json", "w", encoding="utf-8") as f:
    json.dump(cot_infos, f, indent=2, ensure_ascii=False)

len(cot_infos)

3687 Question: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
Thought: Let's think step by step. The American action crime thriller film directed by Stuart Bird is "The Veteran" (2011). Irene Jacob is a French-Swiss actress known for films like "The Double Life of Véronique" (1991) and "Three Colors: Red" (1994). To find the movie she completed before "The Veteran," we need to identify her film immediately prior to 2011. However, Irene Jacob did not star in "The Veteran," so the question might be asking about the last film she completed before the release of "The Veteran."

Looking at her filmography, before 2011, one of her notable films was "L'autre" (1999) and "The Dancer"
{'question_idx': 3687, 'question': 'What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?', 'answer': 'Looking at her filmography, before 2011, one of her notable films was "L\'autre" 

10